# OCR A Level Computer Science: Binary Number Formats and Arithmetic

### Slideshow lesson recap (based on the two original PowerPoints)
- Binary arithmetic and overflow
- Sign and magnitude
- Two's complement
- Floating point representation and normalisation

This notebook is designed for teaching and worked examples with Manim animations.

## Lesson objectives
By the end of this recap, students should be able to:
1. Represent negative integers using sign and magnitude and two's complement.
2. Add and subtract binary values, including using two's complement for subtraction.
3. Explain overflow in fixed-width binary arithmetic.
4. Convert between denary and floating point binary forms.
5. Normalise floating point numbers with positive and negative mantissas.

In [ ]:
# If needed in a fresh environment, uncomment this line and run once.
#%pip install manim

def to_bin(n: int, bits: int = 8) -> str:
    return format(n & ((1 << bits) - 1), f"0{bits}b")

def sign_magnitude(value: int, bits: int = 8) -> str:
    if bits < 2:
        raise ValueError("bits must be at least 2")
    sign = 0 if value >= 0 else 1
    magnitude = abs(value)
    if magnitude > (2 ** (bits - 1) - 1):
        raise ValueError("value out of range for sign and magnitude")
    return str(sign) + format(magnitude, f"0{bits - 1}b")

def twos_complement(value: int, bits: int = 8) -> str:
    min_val = -(2 ** (bits - 1))
    max_val = 2 ** (bits - 1) - 1
    if not (min_val <= value <= max_val):
        raise ValueError("value out of range for two's complement")
    return to_bin(value, bits)

print("Helpers ready.")

## Binary addition and overflow recap
Binary addition follows the same carry logic as denary.

Rules:
- 0 + 0 = 0
- 0 + 1 = 1
- 1 + 1 = 10 (sum 0, carry 1)
- 1 + 1 + 1 = 11 (sum 1, carry 1)

In fixed width arithmetic (for example, 8 bits), extra carry out of the leftmost bit is discarded.

In [ ]:
# Worked example: 8-bit overflow
a, b, bits = 200, 100, 8
raw_sum = a + b
wrapped = raw_sum & ((1 << bits) - 1)

print(f"a = {a:3d} -> {to_bin(a, bits)}")
print(f"b = {b:3d} -> {to_bin(b, bits)}")
print(f"raw sum = {raw_sum} -> {format(raw_sum, '09b')} (needs 9 bits)")
print(f"stored in 8 bits -> {to_bin(wrapped, bits)} = {wrapped}")
print("Overflow occurred because the true sum does not fit in 8 bits.")

## Sign and magnitude
- Leftmost bit is the sign (0 positive, 1 negative).
- Remaining bits store the magnitude.

Example in 8 bits:
- +3 = 00000011
- -3 = 10000011

Issue: ordinary binary addition does not behave nicely with sign and magnitude.

In [ ]:
# Worked example from the lesson: (+3) + (-3) in sign and magnitude
pos3 = int(sign_magnitude(3, 8), 2)
neg3 = int(sign_magnitude(-3, 8), 2)
sum_bits = to_bin(pos3 + neg3, 8)

print(f"+3 (sign-mag): {sign_magnitude(3, 8)}")
print(f"-3 (sign-mag): {sign_magnitude(-3, 8)}")
print(f"binary add result: {sum_bits}")
print("This is not 0, showing why sign and magnitude is awkward for arithmetic hardware.")

## Two's complement
Two's complement is the standard way to represent signed integers in hardware.

To get -x from +x:
1. Write +x in binary
2. Flip all bits (one's complement)
3. Add 1

Range for n bits: $-(2^{n-1})$ to $2^{n-1} - 1$

In [ ]:
# Worked examples: conversion and subtraction using addition
bits = 8
value = -43
print(f"-43 in 8-bit two's complement: {twos_complement(value, bits)}")

# 65 - 43 is done as 65 + (-43)
a = 65
b = -43
result = (a + b) & ((1 << bits) - 1)

print(f"65   -> {twos_complement(65, bits)}")
print(f"-43  -> {twos_complement(-43, bits)}")
print(f"sum  -> {to_bin(result, bits)} = {result}")
print("Ignoring carry out gives the correct answer: 22.")

In [ ]:
# Manim notebook bootstrap: robust on Windows mapped-drive/UNC paths.

from pathlib import Path

import os



ip = get_ipython()



# Use one canonical working path (often UNC on Windows) so Manim path comparisons succeed.

os.chdir(str(Path.cwd().resolve()))



# Register %%manim cell magic directly when needed.

if "manim" not in ip.magics_manager.magics.get("cell", {}):

    from manim.utils.ipython_magic import ManimMagic

    ip.register_magics(ManimMagic)



from manim import config

config["media_dir"] = str(Path.cwd() / "media")



# Embed rendered videos in notebook output to avoid UNC/local file loading issues.

config["media_embed"] = True



print("Manim magic ready. media_dir:", config["media_dir"])

print("Manim media_embed:", config["media_embed"])

## Manim Troubleshooting (Teacher Notes)

If an animation cell fails, check these in order:

1. Run the setup code cell immediately above this slide first.
2. Confirm you are using the correct notebook kernel (the one with `manim` installed).
3. If you see: "The manim module is not an IPython extension", ignore it if the setup cell says `Manim magic ready`.
4. If rendering fails with path/subpath errors on Windows mapped drives, re-run the setup cell (it normalises to a canonical path).
5. Re-run the animation cell. If needed, restart the kernel and run from the top.

Expected success signal:
- You should see `Manim Community v...` and then an embedded video output.

Tip for Codespaces:
- Use this notebook as-is; it already sets `media_dir` to a local `media` folder for predictable outputs.

If render logs show success but video does not play, this is usually a notebook viewer autoplay/policy restriction rather than a Manim render failure.

In [ ]:
%%manim -qm SignMagnitudeDemo

from manim import *
from manim_helpers import align_group_indices_right, maths_text


class SignMagnitudeDemo(Scene):

    def construct(self):
        title = Text("Sign and Magnitude", font_size=40).to_edge(UP)
        self.play(Write(title))

        subtitle = Text("Add +3 and -3 (sign-magnitude)", font_size=30, color=BLUE)
        subtitle.next_to(title, DOWN, buff=0.45)
        self.play(FadeIn(subtitle, shift=UP * 0.1))

        carry = maths_text("Carry:   00000110", font_size=28, color=YELLOW)
        top = maths_text("         00000011", font_size=32)
        bottom = maths_text("       + 10000011", font_size=32)
        answer_hidden = maths_text("       = ........", font_size=32, color=GREEN)
        answer_full = maths_text("       = 10000110", font_size=32, color=GREEN)

        rows = VGroup(carry, top, bottom, answer_hidden).arrange(
            DOWN, aligned_edge=LEFT, buff=0.22
        ).next_to(subtitle, DOWN, buff=0.38).to_edge(LEFT, buff=0.8)
        answer_full.move_to(answer_hidden)

        align_group_indices_right(rows, [0, 1, 2, 3])
        answer_full.move_to(answer_hidden)

        self.play(FadeIn(top, shift=UP * 0.1), FadeIn(bottom, shift=UP * 0.1), run_time=0.55)
        self.play(FadeIn(carry, shift=UP * 0.1), run_time=0.45)
        self.play(FadeIn(answer_hidden, shift=UP * 0.1), run_time=0.35)

        bit_slots = [i for i, ch in enumerate(answer_hidden.text) if ch == "."]
        for idx in reversed(bit_slots):
            self.play(Transform(answer_hidden[idx], answer_full[idx].copy()), run_time=0.24)

        self.play(FadeOut(carry, shift=UP * 0.15), run_time=0.5)

        wrong = Text("This gives -6, not 0", font_size=28, color=RED).next_to(rows, DOWN, buff=0.35)
        why = Text("Sign and magnitude breaks ordinary addition", font_size=26, color=YELLOW).next_to(wrong, DOWN, buff=0.2)
        self.play(FadeIn(wrong, shift=UP * 0.1), FadeIn(why, shift=UP * 0.1), run_time=0.6)

        wrong_box = SurroundingRectangle(wrong, color=RED, buff=0.1)
        self.play(Create(wrong_box), run_time=0.45)
        self.wait(20)


In [7]:
%%manim -qm TwosComplementConversionDemo

from manim import *
from manim_helpers import align_group_indices_right, maths_text


class TwosComplementConversionDemo(Scene):

    def construct(self):
        title = Text("Two's Complement: Finding -77", font_size=40).to_edge(UP)
        self.play(Write(title))

        s2 = VGroup(
            Text("Find -77 in two's complement", font_size=30, color=GREEN),
            Text("Step 1: Write +77 in binary", font_size=28),
            maths_text("+77  =  01001101", font_size=30),
            Text("Step 2: Flip all bits (one's complement)", font_size=28),
            maths_text("Flip  ->  10110010", font_size=30),
            Text("Step 3: Add 1", font_size=28),
            maths_text("10110010 + 1  =  10110011  =  -77", font_size=30, color=YELLOW),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.3).next_to(title, DOWN, buff=0.45).to_edge(LEFT, buff=0.8)

        align_group_indices_right(s2, [2, 4, 6])

        for line in s2:
            self.play(FadeIn(line, shift=RIGHT * 0.15), run_time=0.6)
            self.wait(0.2)

        final_box = SurroundingRectangle(s2[-1], color=GREEN, buff=0.15)
        self.play(Create(final_box))
        self.wait(20)


[05/19/26 11:05:40] INFO     Animation 11 : Using cached data (hash :                          ]8;id=9532252;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532253;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_3277820039_1605147275)                                                     

[05/19/26 11:05:43] INFO     Animation 12 : Using cached data (hash :                          ]8;id=9532258;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532259;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_2514011427_2107678126)                                                     

[05/19/26 11:05:46] INFO     Animation 13 : Using cached data (hash :                          ]8;id=9532264;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532265;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_331530706_1805202589)                                                      

[05/19/26 11:05:50] INFO     Animation 14 : Using cached data (hash :                          ]8;id=9532270;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532271;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_3142935987_737677990)                                                      

[05/19/26 11:05:52] INFO     Animation 15 : Using cached data (hash :                          ]8;id=9532276;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532277;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_3752658419_1500036110)                                                     

[05/19/26 11:05:54] INFO     Animation 16 : Using cached data (hash :                          ]8;id=9532282;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532283;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_952528981_1604649097)                                                      

                    INFO     Combining to Movie file.                                      ]8;id=9532288;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=9532289;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#753\753]8;;\

                    INFO                                                                   ]8;id=9532294;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=9532295;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/workspaces/OCR-A-Level-Computing-Lessons/media/videos/OCR-A                         
                             -Level-Computing-Lessons/720p30/TwosComplementConversionDemo.                         
                             mp4'                                                                                  
                                                                                                                   

                    INFO     Rendered TwosComplementConversionDemo                                     ]8;id=9532300;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=9532301;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/scene/scene.py#278\278]8;;\
                             Played 17 animations                                                                  

In [ ]:
%%manim -qm BinarySubtractionDemo

from manim import *
from manim_helpers import align_lines_right, first_binary_after_equals_idx, maths_text


class BinarySubtractionDemo(Scene):
    def construct(self):
        title = Text("Binary Subtraction: 65 - 43", font_size=40).to_edge(UP)
        self.play(Write(title))

        head = Text("Step 1: Convert 43 to -43 using two's complement", font_size=30, color=BLUE)
        head.next_to(title, DOWN, buff=0.5)
        self.play(FadeIn(head))

        c1 = maths_text("+43     =    00101011", font_size=32)
        c2 = maths_text("Invert  ->   11010100", font_size=32)
        c3 = maths_text("Add 1   ->   11010101", font_size=32, color=YELLOW)
        conv = VGroup(c1, c2, c3).arrange(DOWN, aligned_edge=LEFT, buff=0.3).next_to(head, DOWN, buff=0.35).to_edge(LEFT, buff=0.8)

        align_lines_right((c1, c2, c3))

        for line in (c1, c2, c3):
            self.play(FadeIn(line, shift=UP * 0.1), run_time=0.45)

        tc_box = SurroundingRectangle(c3, color=YELLOW, buff=0.12)
        self.play(Create(tc_box), run_time=0.45)

        step2 = Text("Step 2: Do 65 + (-43)", font_size=30, color=BLUE)
        step2.next_to(conv, DOWN, buff=0.45)
        self.play(FadeIn(step2))

        eq1 = maths_text(" 65    =    01000001", font_size=32)
        eq2 = maths_text("-43    =    11010101", font_size=32)
        eq3 = maths_text("Sum    =  1 00010110", font_size=32)
        eq4 = maths_text("Ignore carry: result = 00010110  =  22", font_size=32, color=GREEN)
        work = VGroup(eq1, eq2, eq3, eq4).arrange(DOWN, aligned_edge=LEFT, buff=0.35).next_to(step2, DOWN, buff=0.35).to_edge(LEFT, buff=0.8)

        align_lines_right((eq1, eq2, eq3, eq4))

        for eq in (eq1, eq2, eq3):
            self.play(FadeIn(eq, shift=UP * 0.1))

        # Find the first binary digit after '=' so spacing normalization does not break carry highlighting.
        carry_idx = first_binary_after_equals_idx(eq3.text)
        carry_mark = SurroundingRectangle(eq3[carry_idx], color=RED, buff=0.05)
        self.play(Create(carry_mark), run_time=0.4)

        self.play(FadeIn(eq4, shift=UP * 0.1))

        result_idx = first_binary_after_equals_idx(eq4.text)
        result_mark = SurroundingRectangle(eq4[result_idx], color=GREEN, buff=0.08)
        self.play(Create(result_mark), run_time=0.5)
        self.wait(20)


Manim Community v0.20.1

[05/19/26 11:05:55] INFO     Animation 0 : Using cached data (hash :                           ]8;id=9532306;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532307;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             1987512680_614822644_223132457)                                                       

[05/19/26 11:05:56] INFO     Animation 1 : Using cached data (hash :                           ]8;id=9532312;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532313;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_582162083_3118246372)                                                      

[05/19/26 11:05:57] INFO     Animation 2 : Using cached data (hash :                           ]8;id=9532318;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=9532319;file:///usr/local/python/3.12.1/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             2538612922_4117877086_2215766245)                                                     

## Floating point representation (teaching model)
In this lesson model, a floating point value is shown as:
- Mantissa (with sign), then
- Exponent (with sign)

Example format used in the slides: mantissa 8 bits + exponent 4 bits.

Key idea for addition/subtraction:
- Equalise exponents first (align binary points),
- then add/subtract mantissas,
- then renormalise.

In [ ]:
# Decode teaching-format floating point values from the slides
def signed_from_bits(bits: str) -> int:
    n = len(bits)
    value = int(bits, 2)
    if bits[0] == '1':
        value -= 1 << n
    return value

def mantissa_fraction(m_bits: str) -> float:
    sign = -1 if m_bits[0] == '1' else 1
    frac_bits = m_bits[1:]
    frac = 0.0
    for i, bit in enumerate(frac_bits, start=1):
        if bit == '1':
            frac += 2 ** (-i)
    return sign * frac

def decode_fp(m_bits: str, e_bits: str) -> float:
    m = mantissa_fraction(m_bits)
    e = signed_from_bits(e_bits)
    return m * (2 ** e)

examples = [
    ("0.1101100", "0100"),
    ("1.0010100", "0011"),
    ("0.1100000", "1110"),
]

for m, e in examples:
    value = decode_fp(m.replace('.', ''), e)
    print(f"{m} {e} -> {value}")

In [ ]:
# ─── OCR A Level answer-verification assertions ───────────────────────────────
# Run this cell to confirm all helper functions match the OCR mark scheme.

# 1. Sign and magnitude: +3 and -3 in 8 bits
assert sign_magnitude( 3, 8) == "00000011"
assert sign_magnitude(-3, 8) == "10000011"

# Adding them gives the wrong answer — that is the whole lesson point
_s3p = int(sign_magnitude( 3, 8), 2)
_s3n = int(sign_magnitude(-3, 8), 2)
_sm_sum = to_bin(_s3p + _s3n, 8)
assert _sm_sum == "10000110", f"SM add: expected 10000110, got {_sm_sum}"
assert _sm_sum != "00000000", "SM +3 + -3 must NOT give 0"

# 2. Two's complement conversion of -77: flip bits of +77, then add 1
assert twos_complement( 77, 8) == "01001101"
_ones_77 = "".join("0" if b == "1" else "1" for b in "01001101")
assert _ones_77 == "10110010"
assert twos_complement(-77, 8) == "10110011"

# 3. Subtraction 65 − 43 = 22 via two's complement addition
assert twos_complement( 65, 8) == "01000001"
assert twos_complement(-43, 8) == "11010101"
_raw = int("01000001", 2) + int("11010101", 2)   # 65 + 213 = 278 (9 bits)
assert _raw == 278
assert to_bin(_raw, 8) == "00010110"             # drop carry → 22
assert int("00010110", 2) == 22

# 4. Floating-point decode: 0.1100000 × 2^1 = 1.5  and  × 2^-2 = 0.1875
assert abs(decode_fp("01100000", "0001") - 1.5)    < 1e-9
assert abs(decode_fp("01100000", "1110") - 0.1875) < 1e-9   # "1110" = −2 in 4-bit TC

# 5. Normalisation rule: positive starts "01", negative starts "10"
def _is_norm(m: str) -> bool:
    return m[:2] in ("01", "10")

assert     _is_norm("01100000")
assert     _is_norm("10011000")
assert not _is_norm("00100000")
assert not _is_norm("11010000")

print("✓ All OCR A Level answer-verification checks passed")


In [ ]:
%%manim -qm FloatingPointNormalisationRoundTrip

from manim import *
from manim_helpers import align_group_indices_right, maths_text


class FloatingPointNormalisationRoundTrip(Scene):
    def construct(self):
        title = Text("Floating Point Normalisation Round Trip", font_size=40).to_edge(UP)
        self.play(Write(title))

        head1 = Text("Step 1: Fixed point to normalised floating point", font_size=30, color=BLUE)
        head1.next_to(title, DOWN, buff=0.45)
        self.play(FadeIn(head1))

        f1 = maths_text("Fixed point        : 1101.0000_ _ _", font_size=30)
        f2 = maths_text("Normalise          : 0.1101000x 2^4", font_size=30)
        f3 = maths_text("Floating point form: 0.1101000 0100", font_size=30, color=YELLOW)
        fixed_to_fp = VGroup(f1, f2, f3).arrange(DOWN, aligned_edge=LEFT, buff=0.25).next_to(head1, DOWN, buff=0.3).to_edge(LEFT, buff=0.8)

        align_group_indices_right(fixed_to_fp, [0, 1, 2])

        for line in fixed_to_fp:
            self.play(FadeIn(line, shift=UP * 0.1), run_time=0.45)

        fp_box = SurroundingRectangle(f3, color=YELLOW, buff=0.1)
        self.play(Create(fp_box), run_time=0.4)

        head2 = Text("Step 2: Floating point back to fixed point", font_size=30, color=BLUE)
        head2.next_to(fixed_to_fp, DOWN, buff=0.45)
        self.play(FadeIn(head2))

        b1 = maths_text("Mantissa + exponent : 0.1101000 and +4", font_size=30)
        b2 = maths_text("Shift point right 4 : 1101.0000", font_size=30)
        b3 = maths_text("Recovered fixed     : 1101.0000", font_size=30, color=GREEN)
        fp_to_fixed = VGroup(b1, b2, b3).arrange(DOWN, aligned_edge=LEFT, buff=0.25).next_to(head2, DOWN, buff=0.3).to_edge(LEFT, buff=0.8)

        align_group_indices_right(fp_to_fixed, [0, 1, 2])

        for line in fp_to_fixed:
            self.play(FadeIn(line, shift=UP * 0.1), run_time=0.45)

        recovered_box = SurroundingRectangle(b3, color=GREEN, buff=0.1)
        self.play(Create(recovered_box), run_time=0.4)
        self.wait(20)


In [ ]:
%%manim -qm FloatingPointAdditionWorkedSteps

from manim import *
from manim_helpers import align_group_indices_right, maths_text


class FloatingPointAdditionWorkedSteps(Scene):

    def construct(self):

        title = Text("Floating Point Addition: Step by Step", font_size=40).to_edge(UP)
        self.play(Write(title))

        problem = maths_text("A = 0.1100000 0001,   B = 0.1111100 0011", font_size=30).shift(UP * 2.0)
        self.play(FadeIn(problem))

        steps = [
            "1) Convert to fixed point by applying each exponent",
            "   A -> 1.1000      B -> 111.1100",
            "2) Add mantissas in fixed point",
            "   1.1000 + 111.1100 = 1001.0100",
            "3) Number is positive, so sign bit = 0",
            "4) Renormalise result",
            "   1001.0100 -> 0.1001010 x 2^4",
            "5) Final normalised floating point",
            "   Result = 0.1001010 0100",
        ]

        lines = VGroup(*[maths_text(s, font_size=28) for s in steps]).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
        lines.next_to(problem, DOWN, buff=0.45).to_edge(LEFT, buff=0.7)

        align_group_indices_right(lines, [1, 3, 6, 8])

        for i, line in enumerate(lines):
            self.play(FadeIn(line, shift=UP * 0.15), run_time=0.55)
            if i in (1, 3, 6, 8):
                box = SurroundingRectangle(line, color=YELLOW, buff=0.1)
                self.play(Create(box), run_time=0.35)
                self.play(FadeOut(box), run_time=0.25)
            self.wait(0.2)

        final_box = SurroundingRectangle(lines[-1], color=GREEN, buff=0.15)
        self.play(Create(final_box))
        self.wait(20)


## Student Practice Space: Random Floating-Point Questions

Use this section for live class questions before revealing the subtraction walkthrough.

Suggested prompts to generate on the fly:
- Convert a floating-point value to denary.
- Normalise an unnormalised mantissa/exponent pair.
- Add two floating-point numbers after converting to fixed point.

Pause here for students to attempt, then run the next worked-example cell.

## Practice Q1: Two's Complement Conversion

**Question:** Convert +45 to -45 in 8-bit two's complement.

<br><br><br><br><br>

## Practice Q1: Two's Complement Conversion

**Question:** Convert +45 to -45 in 8-bit two's complement.

<br><br><br>
**Answer:**
- +45 = 00101101
- Invert -> 11010010
- Add 1 -> 11010011

## Practice Q2: Two's Complement Subtraction

**Question:** Compute 73 - 19 using two's complement addition.

<br><br><br><br><br>

## Practice Q2: Two's Complement Subtraction

**Question:** Compute 73 - 19 using two's complement addition.

<br><br><br>
**Answer:**
- 73 = 01001001
- -19 = 11101101
- Sum = 1 00110110
- Ignore carry -> 00110110 = 54

## Practice Q3: Floating-Point Normalisation

**Question:** Normalise fixed-point 0011.0100 into floating-point form.

<br><br><br><br><br>

## Practice Q3: Floating-Point Normalisation

**Question:** Normalise fixed-point 0011.0100 into floating-point form.

<br><br><br>
**Answer:**
- 0011.0100 = 3.25
- Normalised: 0.1101000 x 2^2
- Floating-point form: 0.1101000 0010

## Practice Q4: Floating-Point Addition

**Question:** Add A = 0.1100000 0001 and B = 0.1010000 0001.

<br><br><br><br><br>

## Practice Q4: Floating-Point Addition

**Question:** Add A = 0.1100000 0001 and B = 0.1010000 0001.

<br><br><br>
**Answer:**
- A -> 1.1000000
- B -> 1.0100000
- Sum -> 10.1100000
- Normalised -> 0.1011000 0010

In [ ]:
import random

random.seed()   # fresh seed each run

def _rand_mantissa(neg=False, bits=8):
    """Normalised mantissa; last bit fixed to 0 so a right-shift is lossless."""
    pfx = "10" if neg else "01"
    mid = "".join(random.choice("01") for _ in range(bits - 3))
    return pfx + mid + "0"

def _rand_exp4(lo=-5, hi=5):
    e = random.randint(lo, hi)
    return format(e & 0xF, "04b"), e

N = 3  # questions per type

print("══ Type 1: Decode floating-point to denary ══")
for _ in range(N):
    m = _rand_mantissa(neg=random.choice([False, True]))
    e_bits, e_val = _rand_exp4()
    val = decode_fp(m, e_bits)
    print(f"  {m[0]}.{m[1:]}  exp {e_bits} ({e_val:+d})  ->  {val:.6g}")

print()
print("══ Type 2: Two's complement subtraction ══")
for _ in range(N):
    a = random.randint(50, 120)
    b = random.randint(10, min(a - 1, 127))
    a_tc   = twos_complement(a,  8)
    neg_tc = twos_complement(-b, 8)
    raw    = int(a_tc, 2) + int(neg_tc, 2)
    result = int(to_bin(raw, 8), 2)
    assert result == a - b, f"{a}-{b}: expected {a-b}, got {result}"
    print(f"  {a} - {b} = ?   [{a}: {a_tc}, -{b}: {neg_tc}]   -> {a-b} = {to_bin(raw, 8)}")

print()
print("══ Type 3: Normalise the following floating-point value ══")
for _ in range(N):
    neg = random.choice([False, True])
    m_norm = _rand_mantissa(neg=neg)
    _, e_norm = _rand_exp4(-4, 4)
    # Shift fractional bits right by 1 (insert 0 after sign, drop LSB) to unnormalise.
    m_unnorm = m_norm[0] + "0" + m_norm[1:-1]
    e_unnorm = e_norm + 1
    assert abs(decode_fp(m_unnorm, format(e_unnorm & 0xF, "04b")) -
               decode_fp(m_norm,   format(e_norm   & 0xF, "04b"))) < 1e-9, "value changed"
    print(f"  Unnorm: {m_unnorm[0]}.{m_unnorm[1:]}  exp {format(e_unnorm&0xF,'04b')} ({e_unnorm:+d})")
    print(f"  -> Norm: {m_norm[0]}.{m_norm[1:]}  exp {format(e_norm&0xF,'04b')} ({e_norm:+d})"
          f"  [value: {decode_fp(m_norm, format(e_norm&0xF,'04b')):.6g}]")
    print()


In [ ]:
%%manim -qm FloatingPointSubtractionWorkedSteps

from manim import *
from manim_helpers import align_group_indices_right, maths_text


class FloatingPointSubtractionWorkedSteps(Scene):

    def construct(self):

        title = Text("Floating Point Subtraction: Step by Step", font_size=40).to_edge(UP)
        self.play(Write(title))

        problem = maths_text("A = 0.1101000 0100,   B = 0.1110000 0011", font_size=30).shift(UP * 2.0)
        self.play(FadeIn(problem))

        steps = [
            "1) Convert each value to fixed point",
            "   A -> 1101.0000     B -> 0111.0000",
            "2) Subtract by adding negative B",
            "   One's complement(B) -> 1000.1111",
            "   Two's complement(B) -> 1001.0000",
            "3) Add A + (-B)",
            "   1101.0000 + 1001.0000 = 0110.0000 (ignore overflow)",
            "4) Convert back to normalised floating point",
            "   Result = 0.1100000 0011",
        ]

        lines = VGroup(*[maths_text(s, font_size=26) for s in steps]).arrange(DOWN, aligned_edge=LEFT, buff=0.26)
        lines.next_to(problem, DOWN, buff=0.45).to_edge(LEFT, buff=0.7)

        align_group_indices_right(lines, [1, 3, 4, 6, 8])

        for i, line in enumerate(lines):
            self.play(FadeIn(line, shift=UP * 0.15), run_time=0.55)
            if i in (1, 3, 4, 6, 8):
                box = SurroundingRectangle(line, color=YELLOW, buff=0.1)
                self.play(Create(box), run_time=0.35)
                self.play(FadeOut(box), run_time=0.25)
            self.wait(0.2)

        final_box = SurroundingRectangle(lines[-1], color=GREEN, buff=0.15)
        self.play(Create(final_box))
        self.wait(20)


## Plenary quick-check
1. Why is two's complement preferred over sign and magnitude in processors?
2. In 8-bit two's complement, what is the range of values?
3. What does overflow mean in fixed-width binary arithmetic?
4. When adding floating point numbers, why must exponents be aligned first?
5. What is the normalised pattern after the binary point for:
   - positive mantissas?
   - negative mantissas?

You can now extend this notebook with your worksheet questions as practice slides.